Merge and clean tables

In [ ]:
import pandas as pd
import numpy as np

deconv = pd.read_csv('/path/to/data/cell_type_deconvolution_PPCG_TME.csv')

immune_cols = ['epithelial', 'endothelial', 'macrophage', 'fibroblast', 'smooth_muscle','plasma', 't_cd8', 't_cd4', 'b', 'mast', 'monocyte',
               'natural_killer', 't_regulatory', 'dendritic']

for col in immune_cols:
    vals = deconv[col]
    print(f"\n{col}:")
    print(f"  mean={vals.mean():.6f}, median={vals.median():.6f}")
    print(f"  max={vals.max():.6f}")
    print(f"  fraction below 0.01: {(vals < 0.01).mean():.3f}")
    print(f"  fraction below 0.001: {(vals < 0.001).mean():.3f}")

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# ── Load files ────────────────────────────────────────────────────────
meta = pd.read_csv(
    '/path/to/data/'
    'metadata_common_PPCG.tsv', sep='\t'
)
deconv = pd.read_csv(
    '/path/to/data/'
    'cell_type_deconvolution_PPCG_TME.csv'
)

# ── Extract patient ID from deconv sample IDs ─────────────────────────
deconv['ppcg_id'] = deconv['PPCG_RNA_Assay_ID'].str.extract(r'(PPCG\d+)')

# ── Join ──────────────────────────────────────────────────────────────
merged = deconv.merge(meta, on='ppcg_id', how='inner')
print(f'Merged shape: {merged.shape}')
print(f'Unique patients: {merged["ppcg_id"].nunique()}')
print(f'Samples per patient (mean): {merged.groupby("ppcg_id").size().mean():.2f}')

# ── Clean 999 missing values ──────────────────────────────────────────
missing_cols = ['gleason_grade_group', 'path_gleason_sum', 
                'damico_risk', 'relapse_ind', 'mets_ind',
                'death_ind', 'psa_at_tumour_collection',
                'clin_t_stage', 'path_t_stage']
for col in missing_cols:
    merged[col] = merged[col].replace(999, np.nan)

# ── Define cell type columns ──────────────────────────────────────────
cell_type_cols = [c for c in deconv.columns 
                  if c not in ['PPCG_RNA_Assay_ID', 'ppcg_id']]
print(f'\nCell type columns: {cell_type_cols}')

# ── Save merged file ──────────────────────────────────────────────────
merged.to_csv('PPCG_deconv_with_metadata.csv', index=False)
print('Saved merged file')

# ── Check sample counts per grade group ──────────────────────────────
print('\nSamples per Gleason grade group after merge:')
print(merged['gleason_grade_group'].value_counts().sort_index())

check the proportion distributions

In [ ]:
import pandas as pd
import numpy as np

merged = pd.read_csv('PPCG_deconv_with_metadata.csv')
cell_type_cols = ['epithelial', 'plasma', 'endothelial', 't_cd8', 
                  't_cd4', 'b', 'macrophage', 'mast', 'monocyte', 
                  'natural_killer', 't_regulatory', 'dendritic', 
                  'fibroblast', 'smooth_muscle']

# Aggregate to patient level first
merged_patient = merged.groupby('ppcg_id')[
    cell_type_cols + ['gleason_grade_group', 'path_gleason_sum',
                      'damico_risk', 'relapse_ind', 'mets_ind',
                      'donor_relapse_interval']
].mean()

# Clean 999s
for col in ['gleason_grade_group', 'path_gleason_sum', 
            'damico_risk', 'relapse_ind', 'mets_ind']:
    merged_patient[col] = merged_patient[col].replace(999, np.nan)

print('=== Cell type proportion summary (patient level) ===')
print(merged_patient[cell_type_cols].describe().round(4).to_string())
print()

# Check which cell types have meaningful variance
print('=== Mean proportions across all patients ===')
means = merged_patient[cell_type_cols].mean().sort_values(ascending=False)
for ct, m in means.items():
    std = merged_patient[ct].std()
    print(f'  {ct}: mean={m:.4f}, std={std:.4f}')

# Save patient-level file for all downstream analyses
merged_patient.to_csv('PPCG_patient_level.csv')
print('\nSaved patient-level file')

from macrophage down all have too small medians, so only look at epithelial, smc, fibrobl, endothel and macrophage

this cell down is just to do the boxplots in two rows instead of one

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns

merged_patient = pd.read_csv('PPCG_patient_level.csv', index_col='ppcg_id')

# Cell types worth testing
testable_cts = ['epithelial', 'smooth_muscle', 'fibroblast', 
                'endothelial', 'macrophage']

# ── Analysis A: Kruskal-Wallis + Spearman vs Gleason grade group ──────
merged_grade = merged_patient.dropna(subset=['gleason_grade_group'])
merged_grade = merged_grade[
    merged_grade['gleason_grade_group'].isin([1,2,3,4,5])
]

print(f'Patients with grade data: {len(merged_grade)}')
print('Grade distribution:')
print(merged_grade['gleason_grade_group'].value_counts().sort_index())

kw_results = []
for ct in testable_cts:
    groups = [
        merged_grade[merged_grade['gleason_grade_group']==g][ct].dropna()
        for g in [1,2,3,4,5]
    ]
    stat, p = stats.kruskal(*groups)
    r, p_spear = stats.spearmanr(
        merged_grade['gleason_grade_group'],
        merged_grade[ct]
    )
    kw_results.append({
        'cell_type': ct,
        'kruskal_stat': round(stat, 3),
        'kruskal_p': round(p, 6),
        'spearman_r': round(r, 3),
        'spearman_p': round(p_spear, 6)
    })

kw_df = pd.DataFrame(kw_results).sort_values('kruskal_p')

# Bonferroni correction
_, p_adj, _, _ = multipletests(kw_df['kruskal_p'], method='bonferroni')
kw_df['p_adj'] = p_adj.round(6)
kw_df['significant'] = kw_df['p_adj'] < 0.05

print('\n=== Kruskal-Wallis: cell type ~ Gleason grade group ===')
print(kw_df.to_string(index=False))
kw_df.to_csv('PPCG_kruskal_gleason.csv', index=False)
# ── Boxplot ───────────────────────────────────────────────────────────
# FIXED: Changed grid architecture to 2 rows and 3 columns, adjusted figsize aspect ratio
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten() # Flattens the 2D grid into a 1D array for sequence looping

grade_labels = {1:'GG1', 2:'GG2', 3:'GG3', 4:'GG4', 5:'GG5'}

for i, ct in enumerate(testable_cts):
    ax = axes[i]  # Safely indexes panels 0, 1, 2 (Row 1) and 3, 4 (Row 2)
    plot_data = merged_grade[['gleason_grade_group', ct]].dropna().copy()
    plot_data['gleason_grade_group'] = plot_data['gleason_grade_group'].astype(int)
    
    sns.boxplot(
        data=plot_data,
        x='gleason_grade_group', y=ct,
        ax=ax, palette='RdYlBu_r',
        order=[1,2,3,4,5]
    )
    
    r = kw_df[kw_df['cell_type']==ct]['spearman_r'].values[0]
    p = kw_df[kw_df['cell_type']==ct]['p_adj'].values[0]
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    ax.set_title(f'{ct}\nr={r:.2f}, Bonferroni p={p:.2e} {sig}', fontsize=10)
    ax.set_xlabel('Gleason Grade Group')
    ax.set_ylabel('Proportion')
    ax.set_xticklabels([grade_labels[g] for g in [1,2,3,4,5]])

# FIXED: Safely clear the empty 6th subplot pane from your thesis visualization canvas
fig.delaxes(axes[5])

plt.suptitle(
    'Cell type proportions across Gleason grade groups\n'
    f'PPCG cohort (n={len(merged_grade)} patients)',
    fontsize=12, fontweight='bold', y=0.98 # Added y-offset margin spacing to prevent title clipping
)
plt.tight_layout()
plt.savefig('PPCG_celltype_gleason_boxplots.png', 
            dpi=150, bbox_inches='tight')
plt.close()
print('Saved boxplot')

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns

merged_patient = pd.read_csv('PPCG_patient_level.csv', index_col='ppcg_id')

# Cell types worth testing
testable_cts = ['epithelial', 'smooth_muscle', 'fibroblast', 
                'endothelial', 'macrophage']

# ── Analysis A: Kruskal-Wallis + Spearman vs Gleason grade group ──────
merged_grade = merged_patient.dropna(subset=['gleason_grade_group'])
merged_grade = merged_grade[
    merged_grade['gleason_grade_group'].isin([1,2,3,4,5])
]

print(f'Patients with grade data: {len(merged_grade)}')
print('Grade distribution:')
print(merged_grade['gleason_grade_group'].value_counts().sort_index())

kw_results = []
for ct in testable_cts:
    groups = [
        merged_grade[merged_grade['gleason_grade_group']==g][ct].dropna()
        for g in [1,2,3,4,5]
    ]
    stat, p = stats.kruskal(*groups)
    r, p_spear = stats.spearmanr(
        merged_grade['gleason_grade_group'],
        merged_grade[ct]
    )
    kw_results.append({
        'cell_type': ct,
        'kruskal_stat': round(stat, 3),
        'kruskal_p': round(p, 6),
        'spearman_r': round(r, 3),
        'spearman_p': round(p_spear, 6)
    })

kw_df = pd.DataFrame(kw_results).sort_values('kruskal_p')

# Bonferroni correction
_, p_adj, _, _ = multipletests(kw_df['kruskal_p'], method='bonferroni')
kw_df['p_adj'] = p_adj.round(6)
kw_df['significant'] = kw_df['p_adj'] < 0.05

print('\n=== Kruskal-Wallis: cell type ~ Gleason grade group ===')
print(kw_df.to_string(index=False))
kw_df.to_csv('PPCG_kruskal_gleason.csv', index=False)

# ── Boxplot ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(testable_cts), figsize=(20, 5))
grade_labels = {1:'GG1', 2:'GG2', 3:'GG3', 4:'GG4', 5:'GG5'}

for i, ct in enumerate(testable_cts):
    ax = axes[i]
    plot_data = merged_grade[['gleason_grade_group', ct]].dropna().copy()
    plot_data['gleason_grade_group'] = plot_data[
        'gleason_grade_group'].astype(int)
    
    sns.boxplot(
        data=plot_data,
        x='gleason_grade_group', y=ct,
        ax=ax, palette='RdYlBu_r',
        order=[1,2,3,4,5]
    )
    
    r = kw_df[kw_df['cell_type']==ct]['spearman_r'].values[0]
    p = kw_df[kw_df['cell_type']==ct]['p_adj'].values[0]
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    ax.set_title(f'{ct}\\nr={r:.2f}, Bonferroni p={p:.2e} {sig}', fontsize=10)
    ax.set_xlabel('Gleason Grade Group')
    ax.set_ylabel('Proportion')
    ax.set_xticklabels([grade_labels[g] for g in [1,2,3,4,5]])

plt.suptitle(
    'Cell type proportions across Gleason grade groups\n'
    f'PPCG cohort (n={len(merged_grade)} patients)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('PPCG_celltype_gleason_boxplots.png', 
            dpi=150, bbox_inches='tight')
plt.close()
print('Saved boxplot')

# ── Analysis B: Mann-Whitney relapse ──────────────────────────────────
merged_relapse = merged_patient.dropna(subset=['relapse_ind'])
merged_relapse = merged_relapse[
    merged_relapse['relapse_ind'].isin([0,1])
]
relapsed = merged_relapse[merged_relapse['relapse_ind']==1]
not_relapsed = merged_relapse[merged_relapse['relapse_ind']==0]

print(f'\nRelapsed: {len(relapsed)}, Not relapsed: {len(not_relapsed)}')

mw_results = []
for ct in testable_cts:
    g1 = relapsed[ct].dropna()
    g2 = not_relapsed[ct].dropna()
    stat, p = stats.mannwhitneyu(g1, g2, alternative='two-sided')
    n1, n2 = len(g1), len(g2)
    effect = 1 - (2 * stat) / (n1 * n2)
    mw_results.append({
        'cell_type': ct,
        'mean_relapsed': round(g1.mean(), 4),
        'mean_not_relapsed': round(g2.mean(), 4),
        'fold_diff': round(g1.mean() / g2.mean(), 3) if g2.mean() > 0 else np.nan,
        'p_val': round(p, 6),
        'effect_size_rb': round(effect, 3)
    })

mw_df = pd.DataFrame(mw_results).sort_values('p_val')
# Bonferroni correction
_, p_adj_mw, _, _ = multipletests(mw_df['p_val'], method='bonferroni')
mw_df['p_adj'] = p_adj_mw.round(6)
mw_df['significant'] = mw_df['p_adj'] < 0.05

print('\n=== Mann-Whitney: cell type ~ relapse ===')
print(mw_df.to_string(index=False))
mw_df.to_csv('PPCG_mannwhitney_relapse.csv', index=False)

In [ ]:
from scikit_posthocs import posthoc_dunn

for ct in ['macrophage', 'smooth_muscle', 'fibroblast']:
    print(f'\n=== Dunn post-hoc: {ct} ===')
    dunn = posthoc_dunn(
        merged_grade, val_col=ct, 
        group_col='gleason_grade_group',
        p_adjust='bonferroni'
    )
    print(dunn.round(4))

GG1→GG3 transition: Normal fibromuscular stroma (smooth muscle) is lost. This is the dominant early stromal event, consistent with LMC8 (normal stroma programme) being epigenomically silenced as cancer progresses.
GG3→GG5 transition: Reactive fibroblasts accumulate and macrophage infiltration increases sharply at GG5. This corresponds to the CAF programme (LMC12) and immune infiltration (LMC13) becoming dominant in advanced disease.
